# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rattan-Kumar/flyrank-ML-Intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    average_precision_score,
    roc_auc_score
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("intership")

con = duckdb.connect()

rel = "hf://datasets/FlyRank/internship-warehouse"

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_token
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

print("FlyRank warehouse connection successful.")

FlyRank warehouse connection successful.


In [5]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available = TRUE
"""

df = con.sql(query).df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 3611061
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']


For Week 5, I will use Logistic Regression as the machine learning method. It fits this problem because the Week 4 baseline is a binary decision: a content item is labeled CTR_FIX_CANDIDATE if it has at least 100 impressions and a CTR below 2%; otherwise, it is labeled NO_ACTION.

Logistic Regression is a simple and easy-to-understand model that can learn how different search-related signals are connected to these decisions. It also gives a probability score, which can be used to rank content items based on how likely they are to be CTR improvement candidates.

I chose to start with a simple model instead of jumping directly to a complex one. The main goal this week is to see whether machine learning can provide useful decision support beyond the transparent Week 4 rule. A more complex model is not automatically better just because it is more advanced.

One important limitation is that the Week 4 label comes directly from the baseline rule, rather than from an independent future outcome. This means the experiment mainly tests how well the ML model can reproduce the existing rule. It does not prove that the pages selected by the model will actually benefit from a CTR improvement.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Calculate CTR (Click-Through Rate)
df["ctr_pct"] = (df["gsc_clicks"] / df["gsc_impressions"]) * 100

# Create the binary target from the Week-4 baseline decision.
# A row is marked as CTR_FIX_CANDIDATE (target=1) when it has at least 100 impressions and CTR below 2%.
# Otherwise, it receives NO_ACTION (target=0).
df["target"] = ((df["gsc_impressions"] >= 100) & (df["ctr_pct"] < 2)).astype(int)

# Use only decision-time numeric signals.
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
]

X = df[feature_columns].copy()
y = df["target"].copy()

# Create a reproducible 80/20 stratified split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training rows: 2888848
Test rows: 722213

Training target distribution:
target
0    0.825828
1    0.174172
Name: proportion, dtype: float64

Test target distribution:
target
0    0.825828
1    0.174172
Name: proportion, dtype: float64


The Week-4 baseline was built on the March 2026 observation set and did not use a train/test split because it was a transparent rule-based ranking.

For Week 5, I will create a reproducible 80/20 stratified split of the same March 2026 rows. The split is used only to test whether Logistic Regression can reproduce the Week-4 decision on rows it did not train on.

The split is not a time-based future validation because the Week-4 baseline itself was not defined with a future outcome window.

The target will be the Week-4 CTR_FIX_CANDIDATE decision. Therefore, the result should be interpreted as reproduction of the baseline rule, not as evidence that a page will actually improve after a refresh.

No client names, URLs, future-window outcomes, or label-derived fields will be used as model features.



## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Build an interpretable Logistic Regression pipeline.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train only on the training data.
model.fit(X_train, y_train)

# Generate probability scores for the held-out test data.
model_scores = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")
print("Test predictions:", len(model_scores))


Logistic Regression trained successfully.
Test predictions: 722213


In [8]:
def precision_at_k(y_true, scores, k=20):
    """
    Calculate the proportion of true positive rows
    inside the top-k ranked rows.
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)[:k]

    return y_true[order].mean()


# Model Precision@20
model_precision_20 = precision_at_k(
    y_test.values,
    model_scores,
    k=20
)

# Recreate the Week-4 baseline score on the same test rows.
baseline_test = df.loc[
    X_test.index,
    [
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "target"
    ]
].copy()

baseline_test["baseline_score"] = (
    baseline_test["gsc_impressions"]
    / (baseline_test["ctr_pct"] + 0.1)
)

baseline_precision_20 = precision_at_k(
    baseline_test["target"].values,
    baseline_test["baseline_score"].values,
    k=20
)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_precision_20,
        model_precision_20
    ]
})

comparison["difference_vs_baseline"] = (
    comparison["Precision@20"]
    - baseline_precision_20
)

comparison

,method,Precision@20,difference_vs_baseline
0,Week-4 baseline,1.0,0.0
1,Logistic Regression,1.0,0.0


I will train the **Logistic Regression** model using the training data and then evaluate it on a separate held-out portion of the data. The model will generate a probability score showing how likely each content item is to be a **`CTR_FIX_CANDIDATE`**, and I will use these scores to create a ranked review queue.

For a fair comparison, I will also calculate the Week 4 baseline score using the exact same held-out rows. The main evaluation metric will be **Precision@20**, since the practical goal is to identify the most relevant items for a small review queue and place the best candidates near the top.

However, the baseline has an important advantage because the target labels were created directly from the Week 4 rule. Therefore, if the ML model does not outperform the baseline, that does not necessarily mean machine learning failed. Instead, it may show that adding a more complex model does not provide measurable value when the goal is simply to reproduce the existing rule.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create an evaluation table for error analysis.
error_results = X_test.copy()

error_results["actual"] = y_test.values
error_results["model_score"] = model_scores
error_results["model_prediction"] = (
    model_scores >= 0.5
).astype(int)

error_results["baseline_score"] = baseline_test["baseline_score"].values

# Identify false positives and false negatives.
false_positives = error_results[
    (error_results["model_prediction"] == 1) &
    (error_results["actual"] == 0)
]

false_negatives = error_results[
    (error_results["model_prediction"] == 0) &
    (error_results["actual"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse-positive examples:")
display(false_positives.head(10))

print("\nFalse-negative examples:")
display(false_negatives.head(10))

False positives: 3016
False negatives: 4225

False-positive examples:


,gsc_impressions,gsc_clicks,ctr_pct,actual,model_score,model_prediction,baseline_score
867140,99,0,0.000000,0,0.564665,1,990.000000
586662,99,0,0.000000,0,0.564665,1,990.000000
2821880,98,0,0.000000,0,0.532603,1,980.000000
2091257,236,5,2.118644,0,0.995641,1,106.371276
1746555,98,0,0.000000,0,0.532603,1,980.000000
1825055,99,0,0.000000,0,0.564665,1,990.000000
642752,99,0,0.000000,0,0.564665,1,990.000000
579222,316,7,2.215190,0,0.999977,1,136.489885
2410511,195,5,2.564103,0,0.544033,1,73.195380
2682861,373,11,2.949062,0,0.999619,1,122.332718



False-negative examples:


,gsc_impressions,gsc_clicks,ctr_pct,actual,model_score,model_prediction,baseline_score
2463988,102,1,0.980392,1,0.142307,0,94.410163
773794,106,1,0.943396,1,0.217071,0,101.591320
2914645,109,2,1.834862,1,0.033885,0,56.334756
2287782,108,2,1.851852,1,0.029953,0,55.332068
2242890,128,2,1.562500,1,0.284231,0,76.992481
3298353,134,2,1.492537,1,0.461273,0,84.142455
1856451,104,1,0.961538,1,0.176596,0,97.971014
411661,106,2,1.886792,1,0.023376,0,53.352327
1680325,103,2,1.941748,1,0.016079,0,50.446981
1764789,127,2,1.574803,1,0.258928,0,75.829807


The main errors are cases where Logistic Regression does not reproduce the Week-4 rule.

False positives are rows that the model ranks as likely CTR_FIX_CANDIDATE but that the Week-4 rule marked as NO_ACTION.

False negatives are rows that the Week-4 rule marked as CTR_FIX_CANDIDATE but that the model gives a lower probability.

I will inspect these errors to understand which search-signal combinations the model handles differently.

I will also inspect the Logistic Regression coefficients. These coefficients describe directional associations learned by the model; they should not be interpreted as causal effects.

Because the target was generated by the Week-4 rule, these errors measure disagreement with the rule rather than prediction of a real future business outcome.



In [10]:
# Extract the learned Logistic Regression coefficients.
logistic_model = model.named_steps["logistic_regression"]

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": logistic_model.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Logistic Regression feature coefficients:")
display(coefficients)

Logistic Regression feature coefficients:


,feature,coefficient,absolute_coefficient
0,gsc_impressions,32.223613,32.223613
1,gsc_clicks,-3.334593,3.334593
2,ctr_pct,0.378475,0.378475


### Interpretation

The model’s errors help us understand where **Logistic Regression** disagrees with the transparent Week 4 rule. Looking at the model coefficients also gives us an idea of which search-related signals have the strongest influence on its predictions.

For this particular task, the Week 4 baseline remains the stronger reference because the target labels were created directly from that same rule. So, the main goal of this experiment is not to claim that ML has found a better content-refresh strategy.

Instead, the experiment is about finding out whether a simple ML model can reproduce the existing decision rule while achieving similar or better ranking quality. If Logistic Regression does not improve **Precision@20**, it suggests that the added complexity of ML may not provide enough value for this specific task.

To properly test whether ML can find better content-refresh opportunities in the future, we would need an independent outcome-based target, such as a leakage-safe measure of whether a page actually improved after the recommended action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.